In [2]:
# Configuração: Imports e Setup dos Parâmetros
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Adicionar path do projeto ao sys.path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

# Carregar os módulos de otimização
exec(open("otimizacao/A_definicao_parametros.py").read(), globals())
exec(open("otimizacao/B_condicoes_modelo.py").read(), globals())
exec(open("otimizacao/C_aproximacao_perdas.py").read(), globals())

# Importar função de visualização
from variaveis_otimizadas.otimo_viz import gerar_figuras

# Importar funções do main
from main import executar_otimizacao

# Constantes
CENARIOS = list(range(11))  # 0 a 10
CONJUNTO_INSTANCIAS_DIR = "conjuntos_instancias"
DIR_SAIDA = "variaveis_otimizadas"

print("✓ Setup concluído")
print(f"  Cenários a analisar: {CENARIOS}")
print(f"  Diretório de dados: {CONJUNTO_INSTANCIAS_DIR}")
print(f"  Diretório de saída: {DIR_SAIDA}")

✓ Setup concluído
  Cenários a analisar: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
  Diretório de dados: conjuntos_instancias
  Diretório de saída: variaveis_otimizadas


In [3]:
# Função de Orquestração: Executar Todos os Cenários
def executar_todos_cenarios(cenarios=CENARIOS, verbose=True):
    """
    Executa a otimização para todos os cenários de microrrede.
    
    Args:
        cenarios (list): Lista de índices dos cenários a executar (default: [0,1,2,3,4,5,6])
        verbose (bool): Se deve exibir logs detalhados
    
    Returns:
        list: Lista com dicionários de resultado para cada cenário
    """
    os.makedirs(DIR_SAIDA, exist_ok=True)
    resultados = []
    
    print(f"\n{'='*70}")
    print(f"EXECUTANDO {len(cenarios)} CENÁRIOS DE OTIMIZAÇÃO")
    print(f"{'='*70}\n")
    
    for idx_cenario in (tqdm(cenarios, desc="Cenários") if not verbose else cenarios):
        # Caminho do arquivo de entrada
        arquivo_entrada = f"{CONJUNTO_INSTANCIAS_DIR}/dados_microrrede_{idx_cenario}.csv"
        
        # Verificar se arquivo existe
        if not os.path.exists(arquivo_entrada):
            print(f"⚠ Arquivo não encontrado: {arquivo_entrada}")
            continue
        
        try:
            # Executar otimização
            resultado = executar_otimizacao(
                caminho_csv=arquivo_entrada,
                pasta_saida_figuras=str(idx_cenario)  # 0, 1, 2, ...
            )
            
            # Adicionar índice do cenário ao resultado
            resultado["cenario"] = idx_cenario
            resultados.append(resultado)
            
        except Exception as e:
            print(f"✗ Erro ao processar cenário {idx_cenario}: {str(e)}")
            continue
    
    print(f"\n{'='*70}")
    print(f"✓ Execução concluída: {len(resultados)}/{len(cenarios)} cenários processados")
    print(f"{'='*70}\n")
    
    return resultados


# Executar todos os cenários
print("Célula pronta para executar: resultados = executar_todos_cenarios()")

Célula pronta para executar: resultados = executar_todos_cenarios()


In [4]:
# Função de Consolidação: Combinar Resultados
def consolidar_resultados(cenarios=CENARIOS):
    """
    Consolida os resultados de todos os cenários em um único DataFrame.
    
    Args:
        cenarios (list): Lista de índices dos cenários
    
    Returns:
        pd.DataFrame: DataFrame consolidado com coluna 'cenario'
    """
    print("\n" + "="*70)
    print("CONSOLIDANDO RESULTADOS")
    print("="*70 + "\n")
    
    dataframes = []
    
    for idx_cenario in cenarios:
        arquivo_csv = f"{DIR_SAIDA}/variaveis_otimizadas_{idx_cenario}.csv"
        
        if not os.path.exists(arquivo_csv):
            print(f"⚠ Arquivo não encontrado: {arquivo_csv}")
            continue
        
        try:
            df = pd.read_csv(arquivo_csv)
            df["cenario"] = idx_cenario
            dataframes.append(df)
            print(f"  ✓ Carregado: {arquivo_csv} ({len(df)} linhas)")
        except Exception as e:
            print(f"  ✗ Erro ao carregar {arquivo_csv}: {str(e)}")
    
    # Concatenar todos os DataFrames
    if dataframes:
        df_consolidado = pd.concat(dataframes, ignore_index=True)
        print(f"\n✓ Consolidação concluída: {len(df_consolidado)} linhas totais")
        
        # Salvar consolidado
        arquivo_consolidado = f"{DIR_SAIDA}/variaveis_otimizadas_consolidado.csv"
        df_consolidado.to_csv(arquivo_consolidado, index=False)
        print(f"✓ Consolidado salvo em: {arquivo_consolidado}\n")
        
        return df_consolidado
    else:
        print("✗ Nenhum arquivo foi carregado!")
        return None


# Função auxiliar para extrair métricas por cenário
def extrair_metricas_cenarios(cenarios=CENARIOS):
    """
    Extrai métricas-chave (custo, energia, perdas) para cada cenário.
    
    Returns:
        pd.DataFrame: DataFrame com resumo de métricas por cenário
    """
    metricas_list = []
    
    for idx_cenario in cenarios:
        arquivo_csv = f"{DIR_SAIDA}/variaveis_otimizadas_{idx_cenario}.csv"
        
        if not os.path.exists(arquivo_csv):
            continue
        
        try:
            df_raw = pd.read_csv(arquivo_csv)
            info = df_raw[df_raw["tempo"] == "INFO"].copy()
            df = df_raw[df_raw["tempo"] != "INFO"].copy()
            
            df["valor"] = pd.to_numeric(df["valor"], errors="coerce")
            
            custo_total = (
                info.loc[info["variavel"] == "custo_total", "valor"]
                .astype(float).iloc[0]
                if (info["variavel"] == "custo_total").any() else np.nan
            )
            
            # Agregar por variável
            pivot = df.pivot_table(index="variavel", values="valor", aggfunc="sum")
            
            metrica = {
                "Cenário": idx_cenario,
                "Custo Total": custo_total,
                "Energia Solar": pivot.loc["PS", "valor"] if "PS" in pivot.index else 0,
                "Energia Eólica": pivot.loc["PW", "valor"] if "PW" in pivot.index else 0,
                "Carga Bateria": pivot.loc["P_ch", "valor"] if "P_ch" in pivot.index else 0,
                "Descarga Bateria": pivot.loc["P_dis", "valor"] if "P_dis" in pivot.index else 0,
                "Perdas Totais": pivot.loc["Ploss", "valor"] if "Ploss" in pivot.index else 0,
            }
            metricas_list.append(metrica)
        except Exception as e:
            print(f"⚠ Erro ao extrair métricas do cenário {idx_cenario}: {str(e)}")
    
    df_metricas = pd.DataFrame(metricas_list)
    return df_metricas


print("Funções de consolidação e extração de métricas definidas")

Funções de consolidação e extração de métricas definidas


In [5]:
# [EXECUÇÃO] Rodar Todos os Cenários de Otimização
print("⏱ Iniciando execução de todos os cenários...")
print("⚠ AVISO: Isso pode levar alguns minutos dependendo da complexidade do modelo\n")

resultados_execucao = executar_todos_cenarios(CENARIOS, verbose=True)

⏱ Iniciando execução de todos os cenários...
⚠ AVISO: Isso pode levar alguns minutos dependendo da complexidade do modelo


EXECUTANDO 11 CENÁRIOS DE OTIMIZAÇÃO


Iniciando otimização para: ./conjuntos_instancias/dados_microrrede_0.csv

[1/4] Carregando parâmetros da microrrede...
  ✓ Carregados: 33 barras, 2 baterias, 32 linhas, 24 períodos

[2/4] Criando e resolvendo modelo...
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/fernando/Documents/1. Projetos/Mestrado/PO201/Modelo/.venv/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/922c9882b8714fd7980dd0a614db564c-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/922c9882b8714fd7980dd0a614db564c-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9485 COLUMNS
At line 37344 RHS
At line 46825 BOUNDS
At line 48434 ENDATA
Problem MODEL has 9480 rows, 11016 columns and 26754 elements
Coin0008I MODEL read with 0 error

In [6]:
# [EXECUÇÃO] Consolidar Resultados e Extrair Métricas
print("Consolidando resultados dos 7 cenários...\n")

# Consolidar todos os resultados em um único arquivo
df_consolidado = consolidar_resultados(CENARIOS)

# Extrair métricas-chave de cada cenário
df_metricas = extrair_metricas_cenarios(CENARIOS)

print("\n✓ Dados consolidados e métricas extraídas com sucesso!")

Consolidando resultados dos 7 cenários...


CONSOLIDANDO RESULTADOS

  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_0.csv (5644 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_1.csv (5644 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_2.csv (5644 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_3.csv (5644 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_4.csv (5500 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_5.csv (5644 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_6.csv (5644 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_7.csv (5644 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_8.csv (5644 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_9.csv (5644 linhas)
  ✓ Carregado: variaveis_otimizadas/variaveis_otimizadas_10.csv (5644 linhas)

✓ Consolidação concluída: 61940 linhas totais
✓ Consolidado salvo em: variaveis_ot

In [ ]:
# [EXECUÇÃO] Gerar Tabelas e Gráficos Comparativos
print("Gerando relatório comparativo...\n")

from variaveis_otimizadas.otimo_viz import (
    gerar_figuras_cenario,
    gerar_analise_consolidada
)

# Celula 2 – Figuras por cenário individual
base_dir = Path("variaveis_otimizadas")
cenarios = list(range(0, 11))  # ajuste se o número de cenários mudar

resultados_cenarios = []

for c in cenarios:
    csv_path = base_dir / f"variaveis_otimizadas_{c}.csv"
    if not csv_path.exists():
        print(f"Arquivo não encontrado para cenário {c}: {csv_path}")
        continue
    print(f"\n=== Gerando figuras para cenário {c} ===")
    res = gerar_figuras_cenario(
        arquivo_csv=str(csv_path),
        pasta_saida=f"figuras/figuras_artigo_cenario_{c}"
    )
    resultados_cenarios.append(res)

# Celula 3 – Análise consolidada entre cenários
csv_consolidado = base_dir / "variaveis_otimizadas_consolidado.csv"
res_global = gerar_analise_consolidada(
    arquivo_csv_consolidado=str(csv_consolidado),
    pasta_saida="figuras/figuras_artigo_global"
)

metricas_cen = res_global["tabela_metricas"]
metricas_cen


Gerando relatório comparativo...


=== Gerando figuras para cenário 0 ===
Lendo arquivo de cenário: variaveis_otimizadas/variaveis_otimizadas_0.csv
  Métricas principais do cenário:
    cenario                         :   0.0000
    Custo total                     : 15328.9678
    Energia solar total             :  13.3418
    Energia eólica total            :  37.9926
    Curtailment solar total         :   0.0000
    Curtailment eólico total        :   2.9658
    Energia carregada bateria       :   0.1805
    Energia descarregada bateria    :   0.0000
    Energia final bateria           :   0.6714
    Perdas acumuladas               :  23.0920

=== Gerando figuras para cenário 1 ===
Lendo arquivo de cenário: variaveis_otimizadas/variaveis_otimizadas_1.csv
  Métricas principais do cenário:
    cenario                         :   0.0000
    Custo total                     : 12976.3563
    Energia solar total             :   9.0683
    Energia eólica total            :  41.8454
    Curt

variavel,curtailment_solar,curtailment_eolica,E,F,E_solar_total,E_eolica_total,E_carga_bat,E_descarga_bat,perdas_totais,theta,custo_total,E_final
cenario,,,,,,,,,,,,
0,0.000000,2.965751,13.801164,-54.088211,13.341754,37.992585,0.180467,0.000000,23.091990,291.609578,15328.967779,0.421444
1,0.000000,2.369135,12.000000,-56.318289,9.068314,41.845446,0.000000,0.000000,23.332101,290.925163,12976.356284,0.250000
2,0.000000,0.007720,13.801164,-24.118571,17.417254,38.236069,0.180467,0.000000,18.773006,181.950185,2848.995558,0.421444
3,0.000000,2.965751,13.801164,-54.088211,13.341754,37.992585,0.180467,0.000000,23.091990,291.609578,15328.967779,0.421444
4,0.000000,2.971901,0.000000,-53.511086,13.341754,37.956565,0.000000,0.000000,22.947542,289.615694,15329.736035,NaN
5,0.002648,3.198320,12.000000,-37.456204,13.403666,40.376452,0.000000,0.000000,20.465743,220.197522,5630.635870,0.250000
6,0.000000,2.847194,12.788854,-75.624835,11.578330,31.708117,1.121440,0.774599,31.306255,442.285756,90139.163520,0.500000
7,0.000000,3.094908,12.000000,-40.126807,13.406314,39.615228,0.000000,0.000000,21.224319,229.153018,21930.576130,0.250000
8,0.000000,0.068413,14.656192,-32.339109,12.635798,44.663592,0.263158,0.000000,17.209631,181.436147,2858.252863,0.250000


In [8]:
# Celula 4 – Exportar tabelas em formato LaTeX (para o artigo)
out_tex_dir = Path("tabelas_artigo")
out_tex_dir.mkdir(exist_ok=True, parents=True)

metricas_cen.to_latex(out_tex_dir / "tabela_metricas_por_cenario.tex",
                      float_format="%.4f")
print("Tabela LaTeX de métricas salva.")

Tabela LaTeX de métricas salva.


In [9]:
# [RELATÓRIO FINAL] Resumo da Análise e Estrutura de Saídas
print("\n" + "="*70)
print("RELATÓRIO FINAL - ANÁLISE DE MÚLTIPLOS CENÁRIOS DE MICRORREDE")
print("="*70 + "\n")

print("📊 ESTRUTURA DE SAÍDAS GERADAS:\n")

print("1. ARQUIVOS CSV:")
print("   └─ variaveis_otimizadas/")
print("      ├─ variaveis_otimizadas_0.csv  (cenário 0)")
print("      ├─ variaveis_otimizadas_1.csv  (cenário 1)")
print("      ├─ ... (cenários 2 até 6)")
print("      └─ variaveis_otimizadas_consolidado.csv  (consolidado com coluna 'cenario')\n")

print("2. PASTAS DE FIGURAS (por cenário):")
print("   └─ figuras_artigo_0/  (cenário 0)")
print("      ├─ fig1_despacho.png")
print("      ├─ fig2_soc_fluxos_perdas.png")
print("      ├─ fig3_acumulado_variaveis.png")
print("      └─ fig4_heatmap_series.png")
print("   ├─ figuras_artigo_1/  (cenário 1)")
print("   ├─ ... (cenários 2 até 6)\n")

print("3. FIGURAS COMPARATIVAS:")
print("   └─ figuras_artigo/")
print("      ├─ comparativo_custo_total.png")
print("      ├─ comparativo_geracao.png")
print("      └─ comparativo_perdas.png\n")

print("="*70)
print("✓ ANÁLISE CONCLUÍDA COM SUCESSO!")
print("="*70 + "\n")

print("💡 PRÓXIMOS PASSOS:")
print("   1. Revise as tabelas de métricas acima")
print("   2. Analise os gráficos comparativos gerados")
print("   3. Consulte as pastas figuras_artigo_X/ para detalhes por cenário")
print("   4. Use variaveis_otimizadas_consolidado.csv para análises adicionais\n")


RELATÓRIO FINAL - ANÁLISE DE MÚLTIPLOS CENÁRIOS DE MICRORREDE

📊 ESTRUTURA DE SAÍDAS GERADAS:

1. ARQUIVOS CSV:
   └─ variaveis_otimizadas/
      ├─ variaveis_otimizadas_0.csv  (cenário 0)
      ├─ variaveis_otimizadas_1.csv  (cenário 1)
      ├─ ... (cenários 2 até 6)
      └─ variaveis_otimizadas_consolidado.csv  (consolidado com coluna 'cenario')

2. PASTAS DE FIGURAS (por cenário):
   └─ figuras_artigo_0/  (cenário 0)
      ├─ fig1_despacho.png
      ├─ fig2_soc_fluxos_perdas.png
      ├─ fig3_acumulado_variaveis.png
      └─ fig4_heatmap_series.png
   ├─ figuras_artigo_1/  (cenário 1)
   ├─ ... (cenários 2 até 6)

3. FIGURAS COMPARATIVAS:
   └─ figuras_artigo/
      ├─ comparativo_custo_total.png
      ├─ comparativo_geracao.png
      └─ comparativo_perdas.png

✓ ANÁLISE CONCLUÍDA COM SUCESSO!

💡 PRÓXIMOS PASSOS:
   1. Revise as tabelas de métricas acima
   2. Analise os gráficos comparativos gerados
   3. Consulte as pastas figuras_artigo_X/ para detalhes por cenário
   4. Use v